In [17]:
import gym
import d4rl
import sys
from pathlib import Path
import numpy as np
from stable_baselines3.ppo import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

PROJECT_ROOT_DIR = Path().absolute().parent
print(PROJECT_ROOT_DIR)

sys.path.append(str(PROJECT_ROOT_DIR))
print(sys.path)

from utils.sb3_env_utils import make_env
from utils.load_data import load_data, load_data_from_my_cache, scale_obs, split_data
from rollout.load_data import load_data as load_data_from_hd5
from utils.sb3_env_wrappers import ScaledObservationWrapper
from models.sb3_model import PPOWithBCLoss
from utils.sb3_evaluate_kl import evaluate_policy_with_kl

/home/ucav/PythonProjects/lorlfd
['/home/ucav/anaconda3/envs/d4rl0/lib/python3.8/site-packages', '/home/ucav/anaconda3/envs/d4rl0/lib/python3.8/site-packages', '/home/ucav/PythonProjects/lorlfd/evaluate', '/home/ucav/PythonProjects/lorlfd/evaluate', '/usr/local/webots/lib/controller/python37', '/home/ucav/dcs/DCS-sdk/dcs-rl-sdk-1.0.x/dcs_sim_sdk', '/home/ucav/anaconda3/envs/d4rl0/lib/python38.zip', '/home/ucav/anaconda3/envs/d4rl0/lib/python3.8', '/home/ucav/anaconda3/envs/d4rl0/lib/python3.8/lib-dynload', '', '/home/ucav/anaconda3/envs/d4rl0/lib/python3.8/site-packages', '/home/ucav/PythonProjects/d4rl-master', '/home/ucav/anaconda3/envs/d4rl0/lib/python3.8/site-packages/pybullet_envs', '/home/ucav/PythonProjects/lorlfd', '/home/ucav/PythonProjects/lorlfd', '/home/ucav/PythonProjects/lorlfd']


In [18]:
ENV = "halfcheetah"
BC_EXPERIMENT_NAME = f"iter_1/medium_{ENV}_256_256_100epochs_loss_1"
RL_BC_EXPERIMENT_NAME = f"iter_1/medium_{ENV}_256_256_1e7steps_8envs_kl1e-1_loss_1"
ENV_NAME = f"{ENV}-medium-v2"
GAMMA = 0.98
# EXPERT_DATA_CACHE_DIR = "rollout/data/iter_1/medium_halfcheetah_256_256_1e7steps_8envs_kl1e-1_loss_1_best_sample.hdf5"
EXPERT_DATA_CACHE_DIR = "cache"

In [19]:
origin_env = gym.make(ENV_NAME)

if EXPERT_DATA_CACHE_DIR == "cache":
    print("load data from d4rl cache.")
    obs, acts, infos = load_data(origin_env)
else:
    data_file: Path = PROJECT_ROOT_DIR / EXPERT_DATA_CACHE_DIR
    print(f"load data from {str(data_file.absolute())}")
    dataset = load_data_from_hd5(str(data_file.absolute()))
    obs, acts, infos = load_data_from_my_cache(dataset)

scaled_obs, scaler = scale_obs(obs)
env = ScaledObservationWrapper(env=origin_env, scaler=scaler)

/home/ucav/anaconda3/envs/d4rl0/lib/python3.8/site-packages/gym/spaces/box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(


load data from d4rl cache.


load datafile: 100%|██████████| 21/21 [00:01<00:00, 14.43it/s]


In [20]:
bc_policy_save_dir = PROJECT_ROOT_DIR / "checkpoints" / "bc" / BC_EXPERIMENT_NAME
bc_ppo = PPOWithBCLoss.load(str((bc_policy_save_dir / "bc_checkpoint").absolute()))

In [21]:
rl_bc_policy_save_dir = PROJECT_ROOT_DIR / "checkpoints" / "rl" / RL_BC_EXPERIMENT_NAME
rl_bc_ppo = PPOWithBCLoss.load(str((rl_bc_policy_save_dir / "best_model").absolute()), env=env)

In [22]:
evaluate_policy_with_kl(model_teacher=bc_ppo, model_student=rl_bc_ppo, sample_model=bc_ppo, env=env, n_eval_episodes=10)

/home/ucav/PythonProjects/lorlfd/utils/sb3_evaluate_kl.py:83: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(4955.641958154365, 148.35873482458496, 7.513389441430569, 1.1056195862580092)

In [23]:
evaluate_policy_with_kl(model_teacher=rl_bc_ppo, model_student=bc_ppo, sample_model=rl_bc_ppo, env=env, n_eval_episodes=10)

/home/ucav/PythonProjects/lorlfd/utils/sb3_evaluate_kl.py:83: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(7908.40808389252, 114.38621523014422, 4.0811045373499395, 1.1526735683415086)

In [24]:
evaluate_policy_with_kl(model_teacher=bc_ppo, model_student=rl_bc_ppo, sample_model=rl_bc_ppo, env=env, n_eval_episodes=10)

/home/ucav/PythonProjects/lorlfd/utils/sb3_evaluate_kl.py:83: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(7897.069488132745, 56.5080489127143, -4.059419433671236, 0.9448068837393075)